# 📊 DataOps Economics Pipeline

A **production-style DataOps project** applying IBM's DataOps Methodology to open macroeconomic data.

This notebook walks through the full pipeline end-to-end:

| Stage | Role | Output |
|-------|------|--------|
| 🔽 **Ingest** | Data Engineer | `raw_indicators` (SQLite) |
| 🔍 **Assess** | DQ Analyst | `quality_scores`, `quality_summary` |
| 🔧 **Remediate** | Data Engineer + DQ Analyst | `refined_indicators` |
| 📐 **Transform** | Data Engineer + Data Scientist | `economic_health_scores` |
| 📚 **Catalog** | Data Steward | `data_catalog` (SQLite + YAML) |
| 📈 **Visualise** | All | Interactive Plotly charts |

**Data source:** [World Bank World Development Indicators](https://databank.worldbank.org/source/world-development-indicators) — free, open, no API key required.

**G20 countries · 5 macroeconomic indicators · 2000–2023**

## ⚙️ 0. Setup — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q pandas requests pyyaml plotly numpy

In [ ]:
import os
import json
import sqlite3
import logging
import warnings
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(name)s] %(message)s')

# Create data directory
os.makedirs('data', exist_ok=True)
os.makedirs('catalog', exist_ok=True)

print('✅ All imports successful!')

## 🔧 1. Configuration (settings.py)

In [ ]:
# ── World Bank indicator codes ────────────────────────────────────────────
INDICATORS = {
    'gdp_per_capita':      'NY.GDP.PCAP.CD',     # GDP per capita (current USD)
    'inflation':           'FP.CPI.TOTL.ZG',     # Inflation, consumer prices (annual %)
    'unemployment':        'SL.UEM.TOTL.ZS',     # Unemployment (% of total labour force)
    'current_account_pct': 'BN.CAB.XOKA.GD.ZS',  # Current account balance (% of GDP)
    'govt_debt_pct':       'GC.DOD.TOTL.GD.ZS',  # Central govt debt (% of GDP)
}

# ── G20 countries (ISO 3166-1 alpha-2) ───────────────────────────────────
G20_COUNTRIES = {
    'AR': 'Argentina',   'AU': 'Australia',      'BR': 'Brazil',
    'CA': 'Canada',      'CN': 'China',           'FR': 'France',
    'DE': 'Germany',     'IN': 'India',           'ID': 'Indonesia',
    'IT': 'Italy',       'JP': 'Japan',           'MX': 'Mexico',
    'RU': 'Russia',      'SA': 'Saudi Arabia',    'ZA': 'South Africa',
    'KR': 'South Korea', 'TR': 'Turkey',          'GB': 'United Kingdom',
    'US': 'United States',
}

# ── Time range ────────────────────────────────────────────────────────────
START_YEAR = 2000
END_YEAR   = 2023

# ── World Bank API ────────────────────────────────────────────────────────
WB_API_BASE  = 'https://api.worldbank.org/v2'
WB_PAGE_SIZE = 1000

# ── SQLite database path ──────────────────────────────────────────────────
DB_PATH = 'data/economics.db'

# ── Data quality thresholds ───────────────────────────────────────────────
QUALITY_THRESHOLDS = {
    'gdp_per_capita':      {'completeness_min': 0.80, 'value_range': (100, 200_000)},
    'inflation':           {'completeness_min': 0.75, 'value_range': (-10, 100)},
    'unemployment':        {'completeness_min': 0.75, 'value_range': (0, 50)},
    'current_account_pct': {'completeness_min': 0.70, 'value_range': (-30, 30)},
    'govt_debt_pct':       {'completeness_min': 0.65, 'value_range': (0, 300)},
}

CATALOG_PATH  = 'catalog/data_catalog.yml'
MAX_INTERP_GAP = 2
SCORE_YEAR_MIN = 2005

print('✅ Configuration loaded')
print(f'   Indicators : {len(INDICATORS)}')
print(f'   Countries  : {len(G20_COUNTRIES)} G20 nations')
print(f'   Time range : {START_YEAR}–{END_YEAR}')

---
## 🔽 Stage 1: Ingest  *(Role: Data Engineer)*

Pulls raw macroeconomic data from the **World Bank API** for all G20 countries.
Writes to `raw_indicators` with **no transformations** — nulls preserved as source truth.

> **DataOps principle:** Separate ingestion from transformation. Keep a pristine raw layer.

In [ ]:
def init_db(conn: sqlite3.Connection) -> None:
    """Create raw_indicators and ingest_log tables."""
    conn.execute("""
        CREATE TABLE IF NOT EXISTS raw_indicators (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            country_code   TEXT    NOT NULL,
            country_name   TEXT    NOT NULL,
            indicator      TEXT    NOT NULL,
            indicator_code TEXT    NOT NULL,
            year           INTEGER NOT NULL,
            value          REAL,
            ingested_at    TEXT    NOT NULL,
            UNIQUE(country_code, indicator, year)
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS ingest_log (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            run_at       TEXT    NOT NULL,
            rows_written INTEGER NOT NULL,
            status       TEXT    NOT NULL,
            notes        TEXT
        )
    """)
    conn.commit()

print('✅ DB init function defined')

In [ ]:
def fetch_indicator(indicator_code: str, country_code: str) -> list:
    """Call the World Bank REST API for one indicator / country pair."""
    url = (
        f"{WB_API_BASE}/country/{country_code}/indicator/{indicator_code}"
        f"?format=json&date={START_YEAR}:{END_YEAR}&per_page={WB_PAGE_SIZE}"
    )
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        payload = response.json()
        if len(payload) < 2 or not payload[1]:
            return []
        return payload[1]
    except requests.RequestException as e:
        print(f'  ⚠️  API error for {indicator_code}/{country_code}: {e}')
        return []


def parse_observations(raw_records: list, indicator_name: str, country_code: str) -> list:
    """Flatten World Bank response into row dicts. Nulls preserved."""
    rows = []
    for rec in raw_records:
        rows.append({
            'country_code':   country_code,
            'country_name':   G20_COUNTRIES.get(country_code, country_code),
            'indicator':      indicator_name,
            'indicator_code': rec.get('indicator', {}).get('id', ''),
            'year':           int(rec.get('date', 0)),
            'value':          rec.get('value'),   # None preserved intentionally
            'ingested_at':    datetime.utcnow().isoformat(),
        })
    return rows


def run_ingestion() -> int:
    """
    Main ingestion entry point.
    Fetches all indicators for all G20 countries and upserts into raw_indicators.
    Returns total rows written.
    """
    conn = sqlite3.connect(DB_PATH)
    init_db(conn)

    all_rows = []
    total = len(G20_COUNTRIES) * len(INDICATORS)
    done  = 0

    print(f'Starting ingestion: {len(G20_COUNTRIES)} countries × {len(INDICATORS)} indicators ({START_YEAR}–{END_YEAR})')
    print(f'Total API calls: {total}\n')

    for country_code, country_name in G20_COUNTRIES.items():
        country_rows = []
        for indicator_name, indicator_code in INDICATORS.items():
            raw = fetch_indicator(indicator_code, country_code)
            rows = parse_observations(raw, indicator_name, country_code)
            country_rows.extend(rows)
            done += 1
        all_rows.extend(country_rows)
        print(f'  [{done:3d}/{total}] ✓ {country_name} — {len(country_rows)} rows')

    if all_rows:
        df = pd.DataFrame(all_rows)
        df.to_sql('raw_indicators', conn, if_exists='append', index=False, method='multi')
        conn.execute("""
            DELETE FROM raw_indicators
            WHERE id NOT IN (
                SELECT MAX(id) FROM raw_indicators
                GROUP BY country_code, indicator, year
            )
        """)
        conn.commit()

    rows_written = len(all_rows)
    conn.execute(
        'INSERT INTO ingest_log (run_at, rows_written, status, notes) VALUES (?,?,?,?)',
        (datetime.utcnow().isoformat(), rows_written, 'SUCCESS',
         f'{len(G20_COUNTRIES)} countries, {len(INDICATORS)} indicators')
    )
    conn.commit()
    conn.close()

    print(f'\n✅ Ingestion complete — {rows_written:,} rows written to raw_indicators')
    return rows_written

print('✅ Ingest functions defined')

In [ ]:
# ▶️ RUN INGESTION  (takes ~2–3 minutes; pulls live data from World Bank API)
rows_written = run_ingestion()

In [ ]:
# Quick look at the raw layer
conn = sqlite3.connect(DB_PATH)
raw_df = pd.read_sql('SELECT * FROM raw_indicators LIMIT 10', conn)
conn.close()

print(f'Shape: {raw_df.shape}')
raw_df

In [ ]:
# Summary by indicator
conn = sqlite3.connect(DB_PATH)
summary = pd.read_sql("""
    SELECT indicator,
           COUNT(*) AS total_rows,
           SUM(CASE WHEN value IS NOT NULL THEN 1 ELSE 0 END) AS non_null,
           ROUND(100.0 * SUM(CASE WHEN value IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS completeness_pct,
           MIN(year) AS year_from,
           MAX(year) AS year_to
    FROM raw_indicators
    GROUP BY indicator
""", conn)
conn.close()
print('Raw Layer — Completeness by Indicator')
summary

---
## 🔍 Stage 2: Assess  *(Role: Data Quality Analyst)*

Runs the **5 Gartner quality dimensions** against every CDE:

| Rule | Dimension | What it checks |
|------|-----------|---------------|
| `completeness_by_country` | Completeness | ≥N% non-null per country per CDE |
| `completeness_overall` | Completeness | Aggregate non-null rate |
| `validity_range_check` | Validity | Values within agreed min/max |
| `uniqueness_check` | Uniqueness | No duplicate (country, indicator, year) |
| `consistency_yoy_change` | Consistency | No >500% year-over-year changes |
| `timeliness_check` | Timeliness | ≥80% countries have most-recent-year data |

> **DataOps principle:** Measure quality *before* acting on data. Thresholds are business agreements.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class QualityResult:
    cde:          str
    rule_name:    str
    dimension:    str
    passed:       bool
    score:        float
    threshold:    float
    exceptions:   int
    total_records: int
    detail:       str = ''
    country_code: Optional[str] = None
    year_range:   tuple = (None, None)
    severity:     Optional[str] = None


class DataQualityRules:
    """All quality rules, tagged by Gartner dimension."""

    def __init__(self, thresholds: dict):
        self.thresholds = thresholds

    def completeness_by_country(self, df: pd.DataFrame, cde: str) -> list:
        """Completeness — per country completeness check."""
        threshold = self.thresholds[cde]['completeness_min']
        results = []
        for country, grp in df.groupby('country_code'):
            total   = len(grp)
            non_null = grp['value'].notna().sum()
            score   = non_null / total if total else 0
            passed  = score >= threshold
            results.append(QualityResult(
                cde=cde, rule_name='completeness_by_country',
                dimension='Completeness', passed=passed,
                score=round(score, 4), threshold=threshold,
                exceptions=total - non_null, total_records=total,
                country_code=country,
                year_range=(int(grp['year'].min()), int(grp['year'].max())),
                detail=f'{non_null}/{total} non-null',
                severity='HIGH' if not passed and score < threshold * 0.8 else ('MEDIUM' if not passed else None)
            ))
        return results

    def completeness_overall(self, df: pd.DataFrame, cde: str) -> list:
        """Completeness — overall non-null rate."""
        threshold = self.thresholds[cde]['completeness_min']
        total    = len(df)
        non_null = df['value'].notna().sum()
        score    = non_null / total if total else 0
        passed   = score >= threshold
        return [QualityResult(
            cde=cde, rule_name='completeness_overall',
            dimension='Completeness', passed=passed,
            score=round(score, 4), threshold=threshold,
            exceptions=total - non_null, total_records=total,
            year_range=(int(df['year'].min()), int(df['year'].max())),
            detail=f'Overall: {non_null}/{total} non-null ({score:.1%})',
            severity='HIGH' if not passed else None
        )]

    def validity_range_check(self, df: pd.DataFrame, cde: str) -> list:
        """Validity — values within agreed min/max range."""
        lo, hi = self.thresholds[cde]['value_range']
        valid_df = df[df['value'].notna()]
        total    = len(valid_df)
        oor      = ((valid_df['value'] < lo) | (valid_df['value'] > hi)).sum()
        score    = (total - oor) / total if total else 1.0
        passed   = oor == 0
        return [QualityResult(
            cde=cde, rule_name='validity_range_check',
            dimension='Validity', passed=passed,
            score=round(score, 4), threshold=1.0,
            exceptions=int(oor), total_records=total,
            year_range=(int(df['year'].min()), int(df['year'].max())),
            detail=f'{oor} values outside [{lo}, {hi}]',
            severity='HIGH' if oor > 0 else None
        )]

    def uniqueness_check(self, df: pd.DataFrame, cde: str) -> list:
        """Uniqueness — no duplicate (country, indicator, year) keys."""
        total = len(df)
        dupes = total - df.drop_duplicates(subset=['country_code', 'indicator', 'year']).shape[0]
        score = (total - dupes) / total if total else 1.0
        return [QualityResult(
            cde=cde, rule_name='uniqueness_check',
            dimension='Uniqueness', passed=(dupes == 0),
            score=round(score, 4), threshold=1.0,
            exceptions=int(dupes), total_records=total,
            year_range=(int(df['year'].min()), int(df['year'].max())),
            detail=f'{dupes} duplicate rows',
            severity='MEDIUM' if dupes > 0 else None
        )]

    def consistency_yoy_change(self, df: pd.DataFrame, cde: str, max_pct: float = 500) -> list:
        """Consistency — flag >500% year-over-year changes as anomalies."""
        results = []
        for country, grp in df.groupby('country_code'):
            grp = grp.sort_values('year')
            pct_change = grp['value'].pct_change().abs() * 100
            anomalies  = (pct_change > max_pct).sum()
            total = len(grp)
            score = (total - anomalies) / total if total else 1.0
            results.append(QualityResult(
                cde=cde, rule_name='consistency_yoy_change',
                dimension='Consistency', passed=(anomalies == 0),
                score=round(score, 4), threshold=1.0,
                exceptions=int(anomalies), total_records=total,
                country_code=country,
                year_range=(int(grp['year'].min()), int(grp['year'].max())),
                detail=f'{anomalies} YoY changes >{max_pct}%',
                severity='MEDIUM' if anomalies > 0 else None
            ))
        return results

    def timeliness_check(self, df: pd.DataFrame, cde: str) -> list:
        """Timeliness — ≥80% countries have data for the most recent year."""
        latest_year = df['year'].max()
        countries_with_latest = df[
            (df['year'] == latest_year) & df['value'].notna()
        ]['country_code'].nunique()
        total_countries = df['country_code'].nunique()
        score   = countries_with_latest / total_countries if total_countries else 0
        threshold = 0.80
        passed  = score >= threshold
        return [QualityResult(
            cde=cde, rule_name='timeliness_check',
            dimension='Timeliness', passed=passed,
            score=round(score, 4), threshold=threshold,
            exceptions=total_countries - countries_with_latest,
            total_records=total_countries,
            year_range=(int(latest_year), int(latest_year)),
            detail=f'{countries_with_latest}/{total_countries} countries have {latest_year} data',
            severity='MEDIUM' if not passed else None
        )]

    def run_all(self, df: pd.DataFrame, cde: str) -> list:
        """Run all rules for a CDE."""
        return (
            self.completeness_by_country(df, cde) +
            self.completeness_overall(df, cde) +
            self.validity_range_check(df, cde) +
            self.uniqueness_check(df, cde) +
            self.consistency_yoy_change(df, cde) +
            self.timeliness_check(df, cde)
        )

print('✅ Quality rules defined')

In [ ]:
def init_quality_tables(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quality_scores (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            assessed_at   TEXT, cde TEXT, rule_name TEXT, dimension TEXT,
            country_code  TEXT, year_from INTEGER, year_to INTEGER,
            passed        INTEGER, score REAL, threshold REAL,
            exceptions    INTEGER, total_records INTEGER,
            detail TEXT, severity TEXT
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS quality_summary (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            assessed_at TEXT, cde TEXT, overall_score REAL,
            rules_passed INTEGER, rules_failed INTEGER,
            total_exceptions INTEGER, status TEXT
        )
    """)
    conn.commit()


def run_assessment() -> list:
    """Run all quality rules for every CDE. Returns summary list."""
    conn = sqlite3.connect(DB_PATH)
    init_quality_tables(conn)
    rules = DataQualityRules(QUALITY_THRESHOLDS)
    assessed_at = datetime.utcnow().isoformat()
    summaries = []

    print(f'Starting assessment for {len(INDICATORS)} CDEs...\n')

    for cde in INDICATORS:
        df = pd.read_sql('SELECT * FROM raw_indicators WHERE indicator = ?',
                         conn, params=(cde,))
        if df.empty:
            print(f'  ⚠️  No data for {cde}')
            continue

        results = rules.run_all(df, cde)

        # Persist scores
        rows = [{
            'assessed_at': assessed_at, 'cde': r.cde, 'rule_name': r.rule_name,
            'dimension': r.dimension, 'country_code': r.country_code,
            'year_from': r.year_range[0], 'year_to': r.year_range[1],
            'passed': int(r.passed), 'score': r.score, 'threshold': r.threshold,
            'exceptions': r.exceptions, 'total_records': r.total_records,
            'detail': r.detail, 'severity': r.severity,
        } for r in results]
        pd.DataFrame(rows).to_sql('quality_scores', conn, if_exists='append', index=False)

        # Summary
        passed      = sum(1 for r in results if r.passed)
        failed      = len(results) - passed
        total_ex    = sum(r.exceptions for r in results)
        overall     = round(sum(r.score for r in results) / len(results), 4) if results else 0
        status      = 'PASS' if failed == 0 else ('WARN' if overall >= 0.70 else 'FAIL')

        conn.execute("""
            INSERT INTO quality_summary
                (assessed_at, cde, overall_score, rules_passed, rules_failed, total_exceptions, status)
            VALUES (?,?,?,?,?,?,?)
        """, (assessed_at, cde, overall, passed, failed, total_ex, status))
        conn.commit()

        icon = '✅' if status == 'PASS' else ('⚠️ ' if status == 'WARN' else '❌')
        print(f'  {icon} {cde:<28} overall={overall:.1%}  rules={passed}/{len(results)} passed  [{status}]')
        summaries.append({'cde': cde, 'overall_score': overall, 'status': status,
                          'rules_passed': passed, 'rules_failed': failed})

    conn.close()
    print('\n✅ Assessment complete')
    return summaries

print('✅ Assessment functions defined')

In [ ]:
# ▶️ RUN ASSESSMENT
assessment_summaries = run_assessment()

In [ ]:
# Visualise quality scores by CDE
conn = sqlite3.connect(DB_PATH)
qs = pd.read_sql("""
    SELECT cde, dimension, AVG(score)*100 AS avg_score_pct
    FROM quality_scores
    GROUP BY cde, dimension
""", conn)
conn.close()

fig = px.bar(
    qs, x='cde', y='avg_score_pct', color='dimension', barmode='group',
    title='Data Quality Scores by CDE and Dimension',
    labels={'avg_score_pct': 'Avg Score (%)', 'cde': 'Critical Data Element'},
    height=400
)
fig.add_hline(y=80, line_dash='dash', line_color='red', annotation_text='80% threshold')
fig.show()

---
## 🔧 Stage 3: Remediate  *(Role: Data Engineer + DQ Analyst)*

Applies targeted cleansing with **full audit logging**:

| Issue | Action |
|-------|--------|
| Duplicate rows | Deduplicated — keep latest ingestion |
| Out-of-range values | **Nulled** and logged — not silently corrected |
| Short gaps (≤2 years) | Linearly interpolated from neighbours |

> **DataOps principle:** Every change is auditable. Root causes are shared with data owners.

In [ ]:
def init_refined_table(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS refined_indicators (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            country_code TEXT, country_name TEXT, indicator TEXT,
            year INTEGER, value_raw REAL, value_refined REAL,
            remediation_applied TEXT, refined_at TEXT,
            UNIQUE(country_code, indicator, year)
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS remediation_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            remediated_at TEXT, cde TEXT, country_code TEXT, year INTEGER,
            issue_type TEXT, original_value REAL, resolved_value REAL,
            action_taken TEXT, notes TEXT
        )
    """)
    conn.commit()


def log_remediation(conn, cde, country, year, issue_type, original, resolved, action, notes=''):
    conn.execute("""
        INSERT INTO remediation_log
            (remediated_at, cde, country_code, year, issue_type,
             original_value, resolved_value, action_taken, notes)
        VALUES (?,?,?,?,?,?,?,?,?)
    """, (datetime.utcnow().isoformat(), cde, country, year,
           issue_type, original, resolved, action, notes))


def remediate_cde(conn, cde: str) -> pd.DataFrame:
    """Apply all remediation steps to one CDE."""
    df = pd.read_sql('SELECT * FROM raw_indicators WHERE indicator = ?',
                     conn, params=(cde,)).copy()
    if df.empty:
        return df

    thresholds = QUALITY_THRESHOLDS[cde]
    lo, hi = thresholds['value_range']
    refined_at = datetime.utcnow().isoformat()

    df['value_raw']     = df['value']
    df['value_refined'] = df['value'].copy()
    df['remediation_applied'] = 'none'

    # Step 1: Deduplication
    before = len(df)
    df = df.sort_values('ingested_at', ascending=False)
    df = df.drop_duplicates(subset=['country_code', 'indicator', 'year'], keep='first')
    dupes = before - len(df)

    # Step 2: Flag out-of-range values (null, don't correct)
    oor_mask = df['value_refined'].notna() & (
        (df['value_refined'] < lo) | (df['value_refined'] > hi)
    )
    for _, row in df[oor_mask].iterrows():
        log_remediation(conn, cde, row['country_code'], row['year'],
                        'out_of_range', row['value_refined'], None,
                        'nulled_flagged', f"Value {row['value_refined']} outside [{lo}, {hi}]")
    df.loc[oor_mask, 'value_refined'] = np.nan
    df.loc[oor_mask, 'remediation_applied'] = 'out_of_range_nulled'

    # Step 3: Interpolate short gaps (≤2 consecutive missing years)
    interp_count = 0
    for country, grp in df.groupby('country_code'):
        grp = grp.sort_values('year').copy()
        if not grp['value_refined'].isna().any():
            continue
        grp['_interp'] = grp['value_refined'].interpolate(
            method='linear', limit=MAX_INTERP_GAP, limit_direction='both'
        )
        newly_filled = grp['_interp'].notna() & grp['value_refined'].isna()
        for idx, row in grp[newly_filled].iterrows():
            log_remediation(conn, cde, country, row['year'],
                            'missing_interpolated', None, row['_interp'],
                            'linear_interpolation',
                            f'Gap ≤{MAX_INTERP_GAP} years — interpolated from neighbours')
            interp_count += 1
        df.loc[grp[newly_filled].index, 'value_refined'] = grp['_interp'][newly_filled].values
        df.loc[grp[newly_filled].index, 'remediation_applied'] = 'interpolated'

    df['refined_at'] = refined_at
    return df[['country_code', 'country_name', 'indicator', 'year',
               'value_raw', 'value_refined', 'remediation_applied', 'refined_at']]


def run_remediation() -> None:
    """Main remediation entry point."""
    conn = sqlite3.connect(DB_PATH)
    init_refined_table(conn)

    print(f'Starting remediation for {len(INDICATORS)} CDEs...\n')
    total_rows = 0

    for cde in INDICATORS:
        refined = remediate_cde(conn, cde)
        if refined.empty:
            print(f'  ⚠️  No data for {cde}')
            continue
        refined.to_sql('refined_indicators', conn, if_exists='append', index=False, method='multi')
        conn.execute("""
            DELETE FROM refined_indicators WHERE id NOT IN (
                SELECT MAX(id) FROM refined_indicators GROUP BY country_code, indicator, year
            )
        """)
        conn.commit()
        total_rows += len(refined)
        actions = refined['remediation_applied'].value_counts().to_dict()
        print(f'  ✅ {cde:<28} {len(refined):>5} rows | actions: {actions}')

    conn.close()
    print(f'\n✅ Remediation complete — {total_rows:,} refined rows written')

print('✅ Remediation functions defined')

In [ ]:
# ▶️ RUN REMEDIATION
run_remediation()

In [ ]:
# Inspect the remediation audit log
conn = sqlite3.connect(DB_PATH)
rem_log = pd.read_sql("""
    SELECT cde, issue_type, COUNT(*) AS count
    FROM remediation_log
    GROUP BY cde, issue_type
    ORDER BY count DESC
""", conn)
conn.close()

print('Remediation Actions Summary')
rem_log

---
## 📐 Stage 4: Transform  *(Role: Data Engineer + Data Scientist)*

Pivots the refined long-format table into a **wide business-ready view** and computes a composite **Economic Health Score (0–100)** per country per year.

| Sub-score | Method |
|-----------|--------|
| GDP score | Log-normalised — higher GDP → higher score |
| Inflation score | Peaks at 2% target, decays symmetrically |
| Unemployment score | 0% → 1.0, 25%+ → 0.0 |
| Current account score | Peaks at 0%, decays for large imbalances |
| Govt debt score | 0% → 1.0, 150%+ GDP → 0.0 |

Health tiers: **Strong** (≥70) · **Moderate** (≥50) · **Weak** (≥30) · **Fragile** (<30)

> **DataOps principle:** Business-ready layer is derived only from cleaned, refined data.

In [ ]:
def score_gdp(series):             return (np.log1p(series.clip(lower=0)) - np.log1p(series.clip(lower=0)).min()) / (np.log1p(series.clip(lower=0)).max() - np.log1p(series.clip(lower=0)).min() + 1e-9)
def score_inflation(series):       return (1.0 / (1.0 + (series - 2.0).abs() / 5.0)).clip(0, 1)
def score_unemployment(series):    return (1.0 - series.clip(0, 25) / 25.0).clip(0, 1)
def score_current_account(series): return (1.0 - series.abs().clip(0, 15) / 15.0).clip(0, 1)
def score_govt_debt(series):       return (1.0 - series.clip(0, 150) / 150.0).clip(0, 1)

def assign_tier(score):
    if pd.isna(score): return 'N/A'
    if score >= 70:    return 'Strong'
    if score >= 50:    return 'Moderate'
    if score >= 30:    return 'Weak'
    return 'Fragile'


def run_transform() -> None:
    """Build the economic_health_scores business-ready table."""
    conn = sqlite3.connect(DB_PATH)

    # Drop + recreate to allow rerun
    conn.execute('DROP TABLE IF EXISTS economic_health_scores')
    conn.execute("""
        CREATE TABLE economic_health_scores (
            id INTEGER PRIMARY KEY AUTOINCREMENT, country_code TEXT, country_name TEXT,
            year INTEGER, gdp_per_capita REAL, inflation REAL, unemployment REAL,
            current_account_pct REAL, govt_debt_pct REAL,
            gdp_score REAL, inflation_score REAL, unemployment_score REAL,
            curr_acc_score REAL, debt_score REAL,
            health_score REAL, health_tier TEXT, indicators_available INTEGER,
            transformed_at TEXT, UNIQUE(country_code, year)
        )
    """)
    conn.commit()

    df = pd.read_sql(
        'SELECT country_code, country_name, indicator, year, value_refined FROM refined_indicators', conn
    )
    wide = df.pivot_table(
        index=['country_code', 'country_name', 'year'],
        columns='indicator', values='value_refined'
    ).reset_index()
    wide.columns.name = None
    for col in INDICATORS:
        if col not in wide.columns:
            wide[col] = np.nan

    wide = wide[wide['year'] >= SCORE_YEAR_MIN].copy()

    wide['gdp_score']          = score_gdp(wide['gdp_per_capita'])
    wide['inflation_score']    = score_inflation(wide['inflation'])
    wide['unemployment_score'] = score_unemployment(wide['unemployment'])
    wide['curr_acc_score']     = score_current_account(wide['current_account_pct'])
    wide['debt_score']         = score_govt_debt(wide['govt_debt_pct'])

    score_cols = ['gdp_score', 'inflation_score', 'unemployment_score', 'curr_acc_score', 'debt_score']
    wide['health_score'] = wide[score_cols].mean(axis=1, skipna=True).mul(100).round(2)
    wide['health_tier']  = wide['health_score'].apply(assign_tier)
    wide['indicators_available'] = wide[list(INDICATORS.keys())].notna().sum(axis=1)
    wide['transformed_at'] = datetime.utcnow().isoformat()

    cols = ['country_code','country_name','year','gdp_per_capita','inflation',
            'unemployment','current_account_pct','govt_debt_pct',
            'gdp_score','inflation_score','unemployment_score','curr_acc_score','debt_score',
            'health_score','health_tier','indicators_available','transformed_at']
    wide[[c for c in cols if c in wide.columns]].to_sql(
        'economic_health_scores', conn, if_exists='replace', index=False
    )
    conn.commit()

    latest_year = int(wide['year'].max())
    top5 = wide[wide['year'] == latest_year].nlargest(5, 'health_score')[['country_name','health_score','health_tier']]

    print(f'✅ Transform complete — {len(wide):,} country-year rows written')
    print(f'\nTop 5 countries by Economic Health Score ({latest_year}):')
    print(top5.to_string(index=False))
    conn.close()

print('✅ Transform functions defined')

In [ ]:
# ▶️ RUN TRANSFORM
run_transform()

---
## 📚 Stage 5: Data Catalog  *(Role: Data Steward)*

Builds a live **data catalog** merging CDE metadata with quality scores and freshness info.

> **DataOps principle:** Metadata, lineage, and governance are first-class citizens — not afterthoughts.

In [ ]:
# CDE definitions (normally loaded from catalog/cde_definitions.yml)
CDE_DEFINITIONS = {
    'gdp_per_capita': {
        'business_name': 'GDP Per Capita (USD)', 'indicator_code': 'NY.GDP.PCAP.CD',
        'domain': 'Macroeconomic Output', 'critical': True,
        'owner': 'Data Engineer', 'privacy_class': 'Public',
        'quality_dims': ['Completeness', 'Validity', 'Timeliness'],
        'thresholds': {'completeness_min': 0.80, 'value_range': [100, 200000]},
        'description': 'GDP per capita in current USD. Primary indicator of economic output per person.',
        'lineage': {'source': 'World Bank WDI API', 'raw': 'raw_indicators',
                    'refined': 'refined_indicators', 'business_ready': 'economic_health_scores'},
    },
    'inflation': {
        'business_name': 'Inflation Rate (Annual %)', 'indicator_code': 'FP.CPI.TOTL.ZG',
        'domain': 'Monetary Stability', 'critical': True,
        'owner': 'Data Engineer', 'privacy_class': 'Public',
        'quality_dims': ['Completeness', 'Validity', 'Consistency'],
        'thresholds': {'completeness_min': 0.75, 'value_range': [-10, 100]},
        'description': 'Consumer price inflation, annual percentage. Key monetary stability metric.',
        'lineage': {'source': 'World Bank WDI API', 'raw': 'raw_indicators',
                    'refined': 'refined_indicators', 'business_ready': 'economic_health_scores'},
    },
    'unemployment': {
        'business_name': 'Unemployment Rate (%)', 'indicator_code': 'SL.UEM.TOTL.ZS',
        'domain': 'Labour Market', 'critical': True,
        'owner': 'Data Engineer', 'privacy_class': 'Public',
        'quality_dims': ['Completeness', 'Validity', 'Timeliness'],
        'thresholds': {'completeness_min': 0.75, 'value_range': [0, 50]},
        'description': 'Unemployment as % of total labour force (modelled ILO estimate).',
        'lineage': {'source': 'World Bank WDI API', 'raw': 'raw_indicators',
                    'refined': 'refined_indicators', 'business_ready': 'economic_health_scores'},
    },
    'current_account_pct': {
        'business_name': 'Current Account Balance (% GDP)', 'indicator_code': 'BN.CAB.XOKA.GD.ZS',
        'domain': 'External Balance', 'critical': False,
        'owner': 'Data Engineer', 'privacy_class': 'Public',
        'quality_dims': ['Completeness', 'Validity'],
        'thresholds': {'completeness_min': 0.70, 'value_range': [-30, 30]},
        'description': 'Current account balance as % of GDP. Measures external trade and transfers.',
        'lineage': {'source': 'World Bank WDI API', 'raw': 'raw_indicators',
                    'refined': 'refined_indicators', 'business_ready': 'economic_health_scores'},
    },
    'govt_debt_pct': {
        'business_name': 'Government Debt (% GDP)', 'indicator_code': 'GC.DOD.TOTL.GD.ZS',
        'domain': 'Fiscal Position', 'critical': False,
        'owner': 'Data Engineer', 'privacy_class': 'Public',
        'quality_dims': ['Completeness', 'Validity'],
        'thresholds': {'completeness_min': 0.65, 'value_range': [0, 300]},
        'description': 'Central government debt as % of GDP. Key fiscal sustainability indicator.',
        'lineage': {'source': 'World Bank WDI API', 'raw': 'raw_indicators',
                    'refined': 'refined_indicators', 'business_ready': 'economic_health_scores'},
    },
}


def run_catalog_build() -> list:
    """Build and persist the data catalog."""
    import yaml
    conn = sqlite3.connect(DB_PATH)
    built_at = datetime.utcnow().isoformat()

    # Latest quality scores
    quality_df = pd.read_sql("""
        SELECT cde, overall_score, rules_passed, rules_failed, total_exceptions, status, assessed_at
        FROM quality_summary
        WHERE assessed_at = (SELECT MAX(assessed_at) FROM quality_summary)
    """, conn)
    quality = {r['cde']: r for r in quality_df.to_dict('records')}

    # Data freshness
    fresh_df = pd.read_sql("""
        SELECT indicator, MAX(year) AS latest_year,
               COUNT(*) AS total_rows,
               SUM(CASE WHEN value_refined IS NOT NULL THEN 1 ELSE 0 END) AS non_null_rows
        FROM refined_indicators GROUP BY indicator
    """, conn)
    freshness = {r['indicator']: r for r in fresh_df.to_dict('records')}

    entries = []
    for cde_key, meta in CDE_DEFINITIONS.items():
        q = quality.get(cde_key, {})
        f = freshness.get(cde_key, {})
        entry = {
            'asset_id': f'econ_pipeline.{cde_key}',
            'business_name': meta['business_name'],
            'technical_name': cde_key,
            'indicator_code': meta['indicator_code'],
            'domain': meta['domain'],
            'description': meta['description'],
            'owner': meta['owner'],
            'privacy_class': meta['privacy_class'],
            'critical_cde': meta['critical'],
            'quality_dims': meta['quality_dims'],
            'completeness_threshold_pct': int(meta['thresholds']['completeness_min'] * 100),
            'value_range': meta['thresholds']['value_range'],
            'quality_score_pct': round(q.get('overall_score', 0) * 100, 1) if q else None,
            'quality_status': q.get('status', 'NOT_ASSESSED'),
            'rules_passed': q.get('rules_passed'),
            'rules_failed': q.get('rules_failed'),
            'total_exceptions': q.get('total_exceptions'),
            'last_assessed': q.get('assessed_at'),
            'latest_year': f.get('latest_year'),
            'total_rows': f.get('total_rows'),
            'non_null_rows': f.get('non_null_rows'),
            'lineage_source': meta['lineage']['source'],
            'lineage_raw': meta['lineage']['raw'],
            'lineage_refined': meta['lineage']['refined'],
            'lineage_business': meta['lineage']['business_ready'],
            'catalog_built_at': built_at,
        }
        entries.append(entry)

    # Persist to SQLite
    conn.execute('DROP TABLE IF EXISTS data_catalog')
    rows = []
    for e in entries:
        row = dict(e)
        row['quality_dims'] = json.dumps(e['quality_dims'])
        row['value_range']  = json.dumps(e['value_range'])
        row['critical_cde'] = int(e['critical_cde'])
        rows.append(row)
    pd.DataFrame(rows).to_sql('data_catalog', conn, if_exists='replace', index=False)
    conn.commit()
    conn.close()

    # Export to YAML
    with open(CATALOG_PATH, 'w') as f:
        yaml.dump({'catalog': entries, 'generated_at': built_at},
                  f, default_flow_style=False, sort_keys=False, allow_unicode=True)

    print(f'✅ Catalog built with {len(entries)} asset entries')
    for e in entries:
        score = e.get('quality_score_pct')
        score_str = f"{score:.1f}%" if score is not None else 'n/a'
        crit = '🔴' if e['critical_cde'] else '🟡'
        print(f'  {crit} {e["business_name"]:<38} quality={score_str}  [{e["quality_status"]}]')

    return entries

print('✅ Catalog functions defined')

In [ ]:
# ▶️ RUN CATALOG BUILD
catalog_entries = run_catalog_build()

In [ ]:
# Display catalog as a DataFrame
conn = sqlite3.connect(DB_PATH)
catalog_df = pd.read_sql("""
    SELECT business_name, domain, critical_cde, quality_score_pct,
           quality_status, completeness_threshold_pct,
           latest_year, total_rows, non_null_rows
    FROM data_catalog
""", conn)
conn.close()

catalog_df['critical_cde'] = catalog_df['critical_cde'].map({1: '✅ Critical', 0: '—'})
catalog_df

---
## 📈 Stage 6: Visualisations — Economic Analysis

Interactive charts from the business-ready layer.

In [ ]:
# Load the business-ready economic health scores
conn = sqlite3.connect(DB_PATH)
scores_df = pd.read_sql('SELECT * FROM economic_health_scores', conn)
conn.close()

print(f'Economic health scores: {scores_df.shape[0]:,} rows  |  {scores_df["country_name"].nunique()} countries  |  {scores_df["year"].min()}–{scores_df["year"].max()}')

In [ ]:
# Economic Health Score — time series for all G20 countries
fig = px.line(
    scores_df, x='year', y='health_score', color='country_name',
    title='G20 Economic Health Scores Over Time (0–100)',
    labels={'health_score': 'Health Score', 'year': 'Year', 'country_name': 'Country'},
    height=500
)
fig.add_hrect(y0=70, y1=100, fillcolor='green',  opacity=0.05, annotation_text='Strong')
fig.add_hrect(y0=50, y1=70,  fillcolor='yellow', opacity=0.05, annotation_text='Moderate')
fig.add_hrect(y0=30, y1=50,  fillcolor='orange', opacity=0.05, annotation_text='Weak')
fig.add_hrect(y0=0,  y1=30,  fillcolor='red',    opacity=0.05, annotation_text='Fragile')
fig.show()

In [ ]:
# Latest year ranking bar chart
latest_year = scores_df['year'].max()
latest = scores_df[scores_df['year'] == latest_year].sort_values('health_score', ascending=True)

tier_colors = {'Strong': '#2ecc71', 'Moderate': '#f39c12', 'Weak': '#e67e22', 'Fragile': '#e74c3c', 'N/A': '#bdc3c7'}

fig = px.bar(
    latest, x='health_score', y='country_name', color='health_tier',
    color_discrete_map=tier_colors,
    title=f'G20 Economic Health Score Ranking — {latest_year}',
    labels={'health_score': 'Health Score (0–100)', 'country_name': 'Country', 'health_tier': 'Tier'},
    height=550, orientation='h'
)
fig.add_vline(x=70, line_dash='dash', line_color='green',  annotation_text='Strong')
fig.add_vline(x=50, line_dash='dash', line_color='orange', annotation_text='Moderate')
fig.add_vline(x=30, line_dash='dash', line_color='red',    annotation_text='Weak')
fig.show()

In [ ]:
# GDP per capita vs Unemployment scatter (latest year)
fig = px.scatter(
    latest.dropna(subset=['gdp_per_capita', 'unemployment']),
    x='gdp_per_capita', y='unemployment', color='health_tier',
    size='health_score', text='country_name',
    color_discrete_map=tier_colors,
    title=f'GDP per Capita vs Unemployment Rate ({latest_year})',
    labels={'gdp_per_capita': 'GDP per Capita (USD)', 'unemployment': 'Unemployment (%)', 'health_tier': 'Tier'},
    height=500, log_x=True
)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
# Completeness heatmap: country × indicator (raw layer)
conn = sqlite3.connect(DB_PATH)
raw_all = pd.read_sql('SELECT country_name, indicator, value FROM raw_indicators', conn)
conn.close()

completeness = raw_all.groupby(['country_name', 'indicator'])['value'].apply(
    lambda s: round(s.notna().mean() * 100, 1)
).reset_index()
completeness.columns = ['country_name', 'indicator', 'completeness_pct']
pivot = completeness.pivot(index='country_name', columns='indicator', values='completeness_pct')

fig = px.imshow(
    pivot, text_auto=True, aspect='auto',
    color_continuous_scale='RdYlGn',
    title='Data Completeness Heatmap — Raw Layer (% non-null)',
    labels={'color': 'Completeness %'},
    height=550
)
fig.show()

In [ ]:
# Inflation trends for selected countries
highlight = ['United States', 'Germany', 'Brazil', 'Turkey', 'India', 'China']
infl = scores_df[scores_df['country_name'].isin(highlight)].dropna(subset=['inflation'])

fig = px.line(
    infl, x='year', y='inflation', color='country_name',
    title='Inflation Rate (%) — Selected G20 Countries',
    labels={'inflation': 'Inflation (%)', 'year': 'Year', 'country_name': 'Country'},
    height=420
)
fig.add_hline(y=2, line_dash='dot', line_color='grey', annotation_text='2% target')
fig.show()

In [ ]:
# Sub-score radar chart for latest year (top 6 countries)
top6 = latest.nlargest(6, 'health_score')['country_name'].tolist()
top6_df = latest[latest['country_name'].isin(top6)]

score_cols = ['gdp_score', 'inflation_score', 'unemployment_score', 'curr_acc_score', 'debt_score']
score_labels = ['GDP', 'Inflation', 'Unemployment', 'Current Account', 'Govt Debt']

fig = go.Figure()
for _, row in top6_df.iterrows():
    vals = [row[c] for c in score_cols]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=score_labels + [score_labels[0]],
        fill='toself', name=row['country_name'], opacity=0.5
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=f'Sub-Score Radar — Top 6 Countries ({latest_year})',
    height=500
)
fig.show()

---
## ✅ Pipeline Summary

A complete DataOps pipeline run. Review the totals below.

In [ ]:
conn = sqlite3.connect(DB_PATH)

tables = {
    'raw_indicators':        'Raw layer',
    'quality_scores':        'Quality rule results',
    'quality_summary':       'Quality summaries',
    'remediation_log':       'Remediation audit log',
    'refined_indicators':    'Refined layer',
    'economic_health_scores':'Business-ready layer',
    'data_catalog':          'Data catalog',
}

print('═' * 55)
print(f'{"Table":<30} {"Rows":>8}   Description')
print('─' * 55)
for table, desc in tables.items():
    count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn)['n'][0]
    print(f'{table:<30} {count:>8,}   {desc}')
print('═' * 55)

conn.close()
print('\n🏁 Pipeline run complete!')

---
## 🚀 Next Steps

- **Run the Streamlit dashboard** locally with `make dashboard` (requires cloning the full repo)
- **Add more indicators** — extend `INDICATORS` in the config cell and re-run
- **Schedule this notebook** using Colab's scheduled runs or Google Cloud Workflows
- **Export to BigQuery** — replace `sqlite3` with `pandas-gbq` for cloud-scale analytics
- **Great Expectations integration** — swap the quality rules module for GX suites

---
*Built with IBM DataOps Methodology · Data sourced from World Bank WDI (free, open, no API key required)*